In [14]:
#collaborative filtering
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [15]:
# 1. Small user–item rating table
ratings = pd.DataFrame({
    'User':  ['A',  'B',  'C'],
    'Item1': [5,    4,    0],
    'Item2': [3,    0,    4],
    'Item3': [0,    2,    5],
}).set_index('User')

print(ratings)

      Item1  Item2  Item3
User                     
A         5      3      0
B         4      0      2
C         0      4      5


In [16]:
# 2. Find how similar users are
user_similarity = cosine_similarity(ratings)
print(user_similarity)

[[1.         0.76696499 0.32140295]
 [0.76696499 1.         0.34921515]
 [0.32140295 0.34921515 1.        ]]


In [17]:
user_similarity = pd.DataFrame(user_similarity,
                               index=ratings.index,
                               columns=ratings.index)
print(user_similarity)

User         A         B         C
User                              
A     1.000000  0.766965  0.321403
B     0.766965  1.000000  0.349215
C     0.321403  0.349215  1.000000


In [18]:
# 3. Simple recommend function
def recommend(user):
    similar_users = user_similarity[user].sort_values(ascending=False)[1:]
    # takes one column (e.g., similarities with user A), sorts from most
    # similar to least, and removes A itself
    scores = {}#store predicted scores for items user A has not rated

    for other_user, sim in similar_users.items():
        for item in ratings.columns:#Check every item
            # only items the target user has not rated
            if ratings.loc[user, item] == 0 and ratings.loc[other_user, item] > 0:
              #Add similarity × rating to that item’s score (weighted sum).
                scores[item] = scores.get(item, 0) + sim * ratings.loc[other_user, item]
    # sort by score (high to low)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)
    # NEW (only top 2)
    #top2 = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:2]
    #return top2

In [19]:
# 4. Show recommendations for user A
print("Recommendations for A:")
for item, score in recommend('A'):
    print(item, "->", round(score, 2))


Recommendations for A:
Item3 -> 3.14
